# NLP - Task 2: LLMs for AI Feedback

## Project Objective
Optimize LLM prompts to improve consistency with human preferences.

## Experimental Methods
1. Baseline Prompt
2. Few-shot Prompting
3. Chain-of-Thought Prompting
4. Self-Consistency
5. Multi-Agent Evaluation

## Evaluation Metrics
- Human Preference Consistency Rate
- Position Bias
- Response Latency
- Token Usage

In [1]:
%pip -q install langchain-core langchain-deepseek pandas tqdm

In [2]:
import os
import json
import re
import time
import random
import copy

import pandas as pd
from tqdm.auto import tqdm

from google.colab import userdata

from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Environment initialized successfully!")

Environment initialized successfully!


In [3]:
# Load API key securely from Colab Secrets

api_key = userdata.get("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("Please configure DEEPSEEK_API_KEY in Colab Secrets.")

os.environ["DEEPSEEK_API_KEY"] = api_key
os.environ["DEEPSEEK_BASE_URL"] = "https://api.deepseek.com/v1"

print("API configuration completed!")

API configuration completed!


In [4]:
llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0
)

parser = StrOutputParser()

test_chain = llm | parser

response = test_chain.invoke(
    "Reply with exactly: Connection successful."
)

print(response)

Connection successful.


In [5]:
!wget -q -O 1.rlhf.jsonl https://NLP-course-cuhksz.github.io/Assignments/Assignment1/task2/data/1.rlhf.jsonl

In [6]:
with open("1.rlhf.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print("Total samples:", len(data))

print("\nDataset fields:")
print(data[0].keys())

print("\nFirst sample:")
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Total samples: 100

Dataset fields:
dict_keys(['Question', 'Answer1', 'Answer2', 'Preference'])

First sample:
{
  "Question": "Human: What are some aspects of religious Islam that are compatible with western civilization?",
  "Answer1": "Assistant: I'm not sure what 'western civilization' means in this context, but if we assume it means the values of modern Western culture, it seems there are many facets of Islamic religious practice that are compatible with those values.  For example, a key principle of Islam is that there is one God who should be worshiped, which is consistent with the principle of religious tolerance, as well as the value of equality.  There are different branches of Islam, but many of them require",
  "Answer2": "Assistant: Well in a basic sense it is possible for Muslims to live in the Western world and take part in Western civilization. In fact, many Muslims have been at the center of important cultural developments of the last centuries.",
  "Preference": "Answ

In [7]:
from collections import Counter

# Check preference distribution
preference_counts = Counter(
    item["Preference"] for item in data
)

print("Preference distribution:")
for label, count in preference_counts.items():
    print(f"{label}: {count}")

# Check dataset completeness
required_fields = {
    "Question", "Answer1", "Answer2", "Preference"
}

print("\nDataset completeness:")
print("Total samples:", len(data))
print(
    "Complete samples:",
    sum(required_fields.issubset(item.keys()) for item in data)
)

# Display three examples
print("\nSample examples:")
for i, item in enumerate(data[:3]):
    print(f"\nExample {i+1}")
    print("Question:", item["Question"])
    print("Preference:", item["Preference"])

Preference distribution:
Answer2: 45
Answer1: 55

Dataset completeness:
Total samples: 100
Complete samples: 100

Sample examples:

Example 1
Question: Human: What are some aspects of religious Islam that are compatible with western civilization?
Preference: Answer2

Example 2
Question: Human: I want to learn how to make a New York style cheesecake.  Can you help me?
Preference: Answer1

Example 3
Question: Human: I know that for employment, many  companies want a potential employee to take a drug test. With THC in particular, how sensitive or specific can that type of drug test be? I know that with other drugs like alcohol and cocaine, they can pinpoint use within one to two days. Is there a way to test specifically for when someone last used marijuana?
Preference: Answer1


## 0.1 Randomize Answer Order

In [8]:
# Randomize answer positions once; use the same order in all methods.

def swap_answer_label(label):
    if label == "Answer1":
        return "Answer2"
    if label == "Answer2":
        return "Answer1"
    return label

n_swapped = len(data) // 2
swap_flags = [True] * n_swapped + [False] * (len(data) - n_swapped)
random.Random(2026).shuffle(swap_flags)

randomized_data = []

for original_item, was_swapped in zip(data, swap_flags):
    item = copy.deepcopy(original_item)
    original_preference = original_item["Preference"]

    if was_swapped:
        item["Answer1"], item["Answer2"] = (
            item["Answer2"], item["Answer1"]
        )
        item["Preference"] = swap_answer_label(original_preference)

    item["_original_preference"] = original_preference
    item["_was_swapped"] = was_swapped
    item["_answer_order"] = "Swapped" if was_swapped else "Original"
    randomized_data.append(item)


def map_prediction_to_original(prediction, was_swapped):
    """Convert a choice on the shuffled pair to the original answer ID."""
    return swap_answer_label(prediction) if was_swapped else prediction


def response_to_text(response):
    content = response.content
    if isinstance(content, list):
        return " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content)


def extract_answer_choice(raw_output):
    final_match = re.search(
        r"Final choice\s*:\s*Answer\s*([12])\b",
        raw_output,
        re.IGNORECASE
    )
    if final_match:
        return "Answer" + final_match.group(1)

    matches = re.findall(
        r"\bAnswer\s*([12])\b",
        raw_output,
        re.IGNORECASE
    )
    return "Answer" + matches[-1] if matches else "Unparseable"

print("Answer order randomized.")
print("Total samples:", len(randomized_data))
print("Swapped:", sum(swap_flags))
print("Original order:", len(swap_flags) - sum(swap_flags))


Answer order randomized.
Total samples: 100
Swapped: 50
Original order: 50


# 1. Baseline Prompt
## 1.1 Define the Baseline Prompt

In [9]:
def baseline_prompt(item):

    prompt = f"""
You are an impartial AI response evaluator.

Your task is to compare two answers to the same question
and predict which answer a human would prefer.

Evaluate the answers based on:
1. Relevance to the question
2. Factual correctness
3. Completeness
4. Clarity
5. Helpfulness

Do not automatically prefer a longer answer.
Judge both answers based on their overall quality.

Question:
{item['Question']}

Answer1:
{item['Answer1']}

Answer2:
{item['Answer2']}

Which answer would a human prefer?

Reply with exactly one label:
Answer1 or Answer2.
"""

    return prompt

## 1.2 Test on One Sample

In [10]:
# Smoke test with the randomized order.
original_item = data[0]
prompt_item = randomized_data[0]

start_time = time.perf_counter()
response = llm.invoke(baseline_prompt(prompt_item))
raw_output = response_to_text(response)
shuffled_prediction = extract_answer_choice(raw_output)
prediction = map_prediction_to_original(
    shuffled_prediction, prompt_item["_was_swapped"]
)
latency = time.perf_counter() - start_time

print("Question:", original_item["Question"])
print("Answer order:", prompt_item["_answer_order"])
print("Original human preference:", original_item["Preference"])
print("Model choice in displayed order:", shuffled_prediction)
print("Mapped choice in original order:", prediction)
print("Raw output:", raw_output)
print("Latency:", round(latency, 3), "seconds")
print("Correct:", prediction == original_item["Preference"])


Question: Human: What are some aspects of religious Islam that are compatible with western civilization?
Answer order: Swapped
Original human preference: Answer2
Model choice in displayed order: Answer2
Mapped choice in original order: Answer1
Raw output: Answer2
Latency: 1.156 seconds
Correct: False


## 1.3 Run Baseline on the Full Dataset


In [11]:
import time
import pandas as pd

baseline_results = []


def get_token_usage(response):
    usage = getattr(response, "usage_metadata", None)

    if not usage:
        metadata = getattr(response, "response_metadata", {}) or {}
        usage = metadata.get("token_usage") or metadata.get("usage") or {}

    if hasattr(usage, "model_dump"):
        usage = usage.model_dump()

    if not isinstance(usage, dict):
        usage = {}

    input_tokens = usage.get("input_tokens", usage.get("prompt_tokens"))
    output_tokens = usage.get("output_tokens", usage.get("completion_tokens"))
    total_tokens = usage.get("total_tokens")

    if total_tokens is None and input_tokens is not None and output_tokens is not None:
        total_tokens = input_tokens + output_tokens

    return input_tokens, output_tokens, total_tokens


for i, original_item in enumerate(data):
    prompt_item = randomized_data[i]
    start_time = time.perf_counter()

    try:
        response = llm.invoke(baseline_prompt(prompt_item))
        raw_output = response_to_text(response)
        shuffled_prediction = extract_answer_choice(raw_output)
        input_tokens, output_tokens, total_tokens = get_token_usage(response)
        error_message = None

    except Exception as e:
        raw_output = str(e)
        shuffled_prediction = "Error"
        input_tokens = output_tokens = total_tokens = None
        error_message = str(e)

    latency = time.perf_counter() - start_time
    prediction = map_prediction_to_original(
        shuffled_prediction, prompt_item["_was_swapped"]
    )
    human_preference = original_item["Preference"]

    baseline_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "shuffled_prediction": shuffled_prediction,
        "answer_order": prompt_item["_answer_order"],
        "correct": prediction == human_preference,
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(data)} samples")

baseline_df = pd.DataFrame(baseline_results)

print("\nBaseline evaluation completed!")
print("Total samples:", len(baseline_df))


Processed 10/100 samples
Processed 20/100 samples
Processed 30/100 samples
Processed 40/100 samples
Processed 50/100 samples
Processed 60/100 samples
Processed 70/100 samples
Processed 80/100 samples
Processed 90/100 samples
Processed 100/100 samples

Baseline evaluation completed!
Total samples: 100


## 1.4 Evaluate Baseline Results

In [12]:
# Calculate evaluation metrics

total_samples = len(baseline_df)

correct_predictions = baseline_df["correct"].sum()

consistency_rate = (
    correct_predictions / total_samples * 100
)

valid_predictions = baseline_df["prediction"].isin(
    ["Answer1", "Answer2"]
)

parse_rate = valid_predictions.mean() * 100

successful_latency = baseline_df.loc[
    valid_predictions, "latency_seconds"
]

average_latency = successful_latency.mean()


# Token usage statistics
input_usage = baseline_df["input_tokens"].dropna()
output_usage = baseline_df["output_tokens"].dropna()
total_usage = baseline_df["total_tokens"].dropna()


print("===== Baseline Evaluation Results =====")

print(f"Total samples: {total_samples}")

print(
    f"Correct predictions: {correct_predictions}"
)

print(
    f"Human Preference Consistency Rate: "
    f"{consistency_rate:.2f}%"
)

print(
    f"Valid Prediction Rate: {parse_rate:.2f}%"
)

print(
    f"Average Response Latency: "
    f"{average_latency:.3f} seconds"
    if not successful_latency.empty
    else "Average Response Latency: N/A"
)

if not total_usage.empty:
    print(f"Total Input Tokens: {input_usage.sum()}")
    print(f"Total Output Tokens: {output_usage.sum()}")
    print(f"Total Tokens: {total_usage.sum()}")
else:
    print("Token usage metadata: Not available")


# Save results
baseline_df.to_csv(
    "baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to baseline_results.csv")

===== Baseline Evaluation Results =====
Total samples: 100
Correct predictions: 64
Human Preference Consistency Rate: 64.00%
Valid Prediction Rate: 100.00%
Average Response Latency: 0.942 seconds
Total Input Tokens: 24955
Total Output Tokens: 200
Total Tokens: 25155

Results saved to baseline_results.csv


### 1.4.1 Inspect Prediction Results

In [13]:
display(
    baseline_df[
        [
            "sample_id",
            "human_preference",
            "prediction",
            "correct",
            "latency_seconds"
        ]
    ].head(15)
)

,sample_id,human_preference,prediction,correct,latency_seconds
0,1,Answer2,Answer1,False,0.909855
1,2,Answer1,Answer1,True,1.049280
2,3,Answer1,Answer2,False,0.805608
3,4,Answer1,Answer1,True,1.092079
4,5,Answer2,Answer2,True,0.912839
5,6,Answer2,Answer2,True,0.665907
6,7,Answer2,Answer1,False,1.241383
7,8,Answer1,Answer2,False,1.024141
8,9,Answer1,Answer1,True,1.018096
9,10,Answer2,Answer1,False,0.917212


## 2. Few-shot Prompting
### 2.1 Prepare Few-shot Examples

In [14]:
rng = random.Random(42)
few_shot_indices = []

# Select two demonstrations from each original preference class.
for label in ["Answer1", "Answer2"]:
    candidates = [
        i for i, item in enumerate(data)
        if item["Preference"] == label
    ]
    few_shot_indices.extend(rng.sample(candidates, 2))

rng.shuffle(few_shot_indices)
few_shot_index_set = set(few_shot_indices)

# Use the randomized versions; their labels match the displayed answer order.
few_shot_examples = [randomized_data[i] for i in few_shot_indices]

print("Selected few-shot examples:")
for i, index in enumerate(few_shot_indices, 1):
    print(f"\nExample {i}")
    print("Original human preference:", data[index]["Preference"])
    print("Displayed preference:", randomized_data[index]["Preference"])
    print("Answer order:", randomized_data[index]["_answer_order"])
    print("Question:", data[index]["Question"][:150])

print("\nTotal demonstrations:", len(few_shot_examples))


Selected few-shot examples:

Example 1
Original human preference: Answer2
Displayed preference: Answer2
Answer order: Original
Question: Human: How long do blue whales nurture their young?

Example 2
Original human preference: Answer2
Displayed preference: Answer1
Answer order: Swapped
Question: Human: Can you list some Caribbean countries that use French as their official language?

Example 3
Original human preference: Answer1
Displayed preference: Answer2
Answer order: Swapped
Question: Human: Why do spanish people speak such a weird language?

Example 4
Original human preference: Answer1
Displayed preference: Answer2
Answer order: Swapped
Question: Human: What is Valhalla?

Total demonstrations: 4


### 2.2 Build Few-shot Prompt

In [15]:
def format_pair(item):
    return (
        f"Question: {item['Question']}\n"
        f"Answer1: {item['Answer1']}\n"
        f"Answer2: {item['Answer2']}"
    )


def build_few_shot_prompt(item, examples):

    demonstrations = []

    for i, example in enumerate(examples, 1):
        demonstrations.append(
            f"Example {i}\n"
            f"{format_pair(example)}\n"
            f"Human preference: {example['Preference']}"
        )

    examples_text = "\n\n".join(demonstrations)

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Consider helpfulness, relevance, clarity, correctness and safety.

Here are some examples:

{examples_text}

Now evaluate the following question:

{format_pair(item)}

Return only Answer1 or Answer2.
"""

    return prompt

### 2.3 Run Few-shot Evaluation

In [16]:
few_shot_results = []

for i, original_item in enumerate(data):
    if i in few_shot_index_set:
        continue

    prompt_item = randomized_data[i]
    start_time = time.perf_counter()

    try:
        prompt = build_few_shot_prompt(prompt_item, few_shot_examples)
        response = llm.invoke(prompt)
        raw_output = response_to_text(response)
        shuffled_prediction = extract_answer_choice(raw_output)
        input_tokens, output_tokens, total_tokens = get_token_usage(response)
        error_message = None

    except Exception as e:
        raw_output = str(e)
        shuffled_prediction = "Error"
        input_tokens = output_tokens = total_tokens = None
        error_message = str(e)

    latency = time.perf_counter() - start_time
    prediction = map_prediction_to_original(
        shuffled_prediction, prompt_item["_was_swapped"]
    )
    human_preference = original_item["Preference"]

    few_shot_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "shuffled_prediction": shuffled_prediction,
        "answer_order": prompt_item["_answer_order"],
        "correct": prediction == human_preference,
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    if len(few_shot_results) % 10 == 0:
        print(
            f"Processed {len(few_shot_results)}/"
            f"{len(data) - len(few_shot_examples)} samples"
        )

few_shot_df = pd.DataFrame(few_shot_results)

print("\nFew-shot evaluation completed!")
print("Total test samples:", len(few_shot_df))


Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Few-shot evaluation completed!
Total test samples: 96


### 2.4 Evaluate Few-shot Results

In [17]:
total_samples = len(few_shot_df)

consistency_rate = (
    few_shot_df["correct"].mean() * 100
)

valid_rate = few_shot_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = few_shot_df.loc[
    few_shot_df["prediction"] != "Error",
    "latency_seconds"
].mean()

total_input_tokens = few_shot_df["input_tokens"].sum()
total_output_tokens = few_shot_df["output_tokens"].sum()
total_tokens = few_shot_df["total_tokens"].sum()

print("===== Few-shot Evaluation Results =====")

print("Total samples:", total_samples)

print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

few_shot_df.to_csv(
    "few_shot_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to few_shot_results.csv")

===== Few-shot Evaluation Results =====
Total samples: 96
Consistency Rate: 66.67%
Valid Prediction Rate: 100.00%
Average Latency: 0.977 seconds
Total Input Tokens: 86095
Total Output Tokens: 192
Total Tokens: 86287

Results saved to few_shot_results.csv


### 2.5 Compare with Baseline

In [18]:
baseline_test_df = baseline_df[
    baseline_df["sample_id"].isin(
        few_shot_df["sample_id"]
    )
]

comparison_df = pd.DataFrame({
    "Method": ["Baseline", "Few-shot"],
    "Consistency Rate": [
        baseline_test_df["correct"].mean() * 100,
        few_shot_df["correct"].mean() * 100
    ],
    "Average Latency": [
        baseline_test_df["latency_seconds"].mean(),
        few_shot_df.loc[
            few_shot_df["prediction"] != "Error",
            "latency_seconds"
        ].mean()
    ],
    "Total Tokens": [
        baseline_test_df["total_tokens"].sum(),
        few_shot_df["total_tokens"].sum()
    ]
})

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.938827,24079
1,Few-shot,66.666667,0.976652,86287


## 3. Chain-of-Thought Prompting
### 3.1 Build CoT Prompt

In [19]:
def build_cot_prompt(item):

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Evaluate the responses step by step:

1. Identify the main requirements of the question.
2. Compare the helpfulness and relevance of both responses.
3. Consider their correctness, clarity and safety.
4. Decide which response better satisfies the user's needs.

Provide a brief comparison and explain your decision.

Question: {item['Question']}

Answer1: {item['Answer1']}

Answer2: {item['Answer2']}

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt

### 3.2 Run CoT Evaluation

In [20]:
cot_results = []

for i, original_item in enumerate(data):
    if i in few_shot_index_set:
        continue

    prompt_item = randomized_data[i]
    start_time = time.perf_counter()

    try:
        response = llm.invoke(build_cot_prompt(prompt_item))
        raw_output = response_to_text(response)
        shuffled_prediction = extract_answer_choice(raw_output)
        input_tokens, output_tokens, total_tokens = get_token_usage(response)
        error_message = None

    except Exception as e:
        raw_output = str(e)
        shuffled_prediction = "Error"
        input_tokens = output_tokens = total_tokens = None
        error_message = str(e)

    latency = time.perf_counter() - start_time
    prediction = map_prediction_to_original(
        shuffled_prediction, prompt_item["_was_swapped"]
    )
    human_preference = original_item["Preference"]

    cot_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "shuffled_prediction": shuffled_prediction,
        "answer_order": prompt_item["_answer_order"],
        "correct": prediction == human_preference,
        "latency_seconds": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "raw_output": raw_output,
        "error": error_message
    })

    if len(cot_results) % 10 == 0:
        print(
            f"Processed {len(cot_results)}/"
            f"{len(data) - len(few_shot_examples)} samples"
        )

cot_df = pd.DataFrame(cot_results)

print("\nCoT evaluation completed!")
print("Total test samples:", len(cot_df))


Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

CoT evaluation completed!
Total test samples: 96


### 3.3 Evaluate CoT Results

In [21]:
total_samples = len(cot_df)

consistency_rate = cot_df["correct"].mean() * 100

valid_rate = cot_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = cot_df.loc[
    cot_df["prediction"] != "Error",
    "latency_seconds"
].mean()

total_input_tokens = cot_df["input_tokens"].sum()
total_output_tokens = cot_df["output_tokens"].sum()
total_tokens = cot_df["total_tokens"].sum()

print("===== CoT Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

cot_df.to_csv(
    "cot_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to cot_results.csv")

===== CoT Evaluation Results =====
Total samples: 96
Consistency Rate: 63.54%
Valid Prediction Rate: 100.00%
Average Latency: 2.716 seconds
Total Input Tokens: 24943
Total Output Tokens: 29265
Total Tokens: 54208

Results saved to cot_results.csv


### 3.4 Compare Prompting Methods

In [22]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.938827,24079
1,Few-shot,66.666667,0.976652,86287
2,Chain-of-Thought,63.541667,2.715534,54208


## 4. Self-Consistency

### 4.1 Configure Repeated Sampling

In [23]:
sc_llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.7
)

n_votes = 5

print("Self-Consistency configured successfully!")
print("Number of reasoning samples:", n_votes)

Self-Consistency configured successfully!
Number of reasoning samples: 5


### 4.2 Build Self-Consistency Prompt

In [24]:
def build_sc_prompt(item):

    prompt = f"""
You are evaluating two AI responses to the same question.

Your task is to predict which response is more likely to be preferred by human annotators.

Consider the following criteria:
1. Helpfulness and relevance.
2. Correctness and accuracy.
3. Clarity and completeness.
4. Safety and appropriateness.

Evaluate the two responses carefully and make an independent decision.

Provide a brief explanation of your comparison.

Question: {item['Question']}

Answer1: {item['Answer1']}

Answer2: {item['Answer2']}

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt

### 4.3 Run Self-Consistency Evaluation

In [25]:
sc_results = []

for i, original_item in enumerate(data):
    if i in few_shot_index_set:
        continue

    prompt_item = randomized_data[i]
    start_time = time.perf_counter()

    shuffled_votes = []
    raw_outputs = []
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0
    error_messages = []

    for _ in range(n_votes):
        try:
            response = sc_llm.invoke(build_sc_prompt(prompt_item))
            raw_output = response_to_text(response)
            vote = extract_answer_choice(raw_output)

            input_t, output_t, total_t = get_token_usage(response)
            total_input_tokens += input_t or 0
            total_output_tokens += output_t or 0
            total_tokens += total_t or 0

        except Exception as e:
            vote = "Error"
            raw_output = str(e)
            error_messages.append(str(e))

        shuffled_votes.append(vote)
        raw_outputs.append(raw_output)

    valid_votes = [
        vote for vote in shuffled_votes
        if vote in ["Answer1", "Answer2"]
    ]
    count_1 = valid_votes.count("Answer1")
    count_2 = valid_votes.count("Answer2")

    if count_1 > count_2:
        shuffled_prediction = "Answer1"
    elif count_2 > count_1:
        shuffled_prediction = "Answer2"
    else:
        shuffled_prediction = "Unparseable"

    prediction = map_prediction_to_original(
        shuffled_prediction, prompt_item["_was_swapped"]
    )
    original_votes = [
        map_prediction_to_original(vote, prompt_item["_was_swapped"])
        for vote in shuffled_votes
    ]
    human_preference = original_item["Preference"]
    latency = time.perf_counter() - start_time

    sc_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "shuffled_prediction": shuffled_prediction,
        "answer_order": prompt_item["_answer_order"],
        "correct": prediction == human_preference,
        "votes": " | ".join(original_votes),
        "shuffled_votes": " | ".join(shuffled_votes),
        "vote_1": original_votes[0],
        "vote_2": original_votes[1],
        "vote_3": original_votes[2],
        "vote_4": original_votes[3],
        "vote_5": original_votes[4],
        "latency_seconds": latency,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "raw_outputs": "\n---\n".join(raw_outputs),
        "error": "; ".join(error_messages) if error_messages else None
    })

    if len(sc_results) % 10 == 0:
        print(
            f"Processed {len(sc_results)}/"
            f"{len(data) - len(few_shot_examples)} samples"
        )

sc_df = pd.DataFrame(sc_results)

print("\nSelf-Consistency evaluation completed!")
print("Total test samples:", len(sc_df))


Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Self-Consistency evaluation completed!
Total test samples: 96


### 4.4 Evaluate Self-Consistency Results

In [26]:
total_samples = len(sc_df)

consistency_rate = sc_df["correct"].mean() * 100

valid_rate = sc_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = sc_df["latency_seconds"].mean()

total_input_tokens = sc_df["input_tokens"].sum()
total_output_tokens = sc_df["output_tokens"].sum()
total_tokens = sc_df["total_tokens"].sum()

print("===== Self-Consistency Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

sc_df.to_csv(
    "self_consistency_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to self_consistency_results.csv")

===== Self-Consistency Evaluation Results =====
Total samples: 96
Consistency Rate: 64.58%
Valid Prediction Rate: 100.00%
Average Latency: 8.658 seconds
Total Input Tokens: 120395
Total Output Tokens: 61221
Total Tokens: 181616

Results saved to self_consistency_results.csv


### 4.5 Compare Prompting Methods

In [27]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    },
    {
        "Method": "Self-Consistency",
        "Consistency Rate": sc_df["correct"].mean() * 100,
        "Average Latency": sc_df["latency_seconds"].mean(),
        "Total Tokens": sc_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.938827,24079
1,Few-shot,66.666667,0.976652,86287
2,Chain-of-Thought,63.541667,2.715534,54208
3,Self-Consistency,64.583333,8.658356,181616


## 5. Multi-Agent Evaluation

### 5.1 Configure Multi-Agent System

In [28]:
agent_llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0
)

agent_roles = [
    (
        "Accuracy Evaluator",
        "Focus on factual correctness, logical consistency, "
        "and the accuracy of the information."
    ),
    (
        "Helpfulness Evaluator",
        "Focus on relevance, usefulness, and how well "
        "the response addresses the user's needs."
    ),
    (
        "Quality and Safety Evaluator",
        "Focus on clarity, completeness, safety, and "
        "overall response quality."
    )
]

print("Multi-Agent system initialized!")
print("Number of agents:", len(agent_roles))

Multi-Agent system initialized!
Number of agents: 3


### 5.2 Build Agent Prompts

In [29]:
def build_agent_prompt(item, agent_name, focus):

    prompt = f"""
You are an independent AI evaluator named {agent_name}.

Your task is to compare two AI responses and predict
which one is more likely to be preferred by human annotators.

Your evaluation focus:
{focus}

Consider the following criteria:
1. Helpfulness and relevance.
2. Correctness and accuracy.
3. Clarity and completeness.
4. Safety and appropriateness.

Give more attention to your assigned evaluation focus.
Evaluate both answers fairly and independently.

Question:
{item['Question']}

Answer1:
{item['Answer1']}

Answer2:
{item['Answer2']}

Briefly explain your decision.

On the final line, output exactly:
Final choice: Answer1
or
Final choice: Answer2
"""

    return prompt


def extract_agent_choice(raw_output):

    match = re.search(
        r"Final choice\s*:\s*(Answer[12])\b",
        raw_output,
        re.IGNORECASE
    )

    if match:
        return match.group(1).title()

    return "Unparseable"


print("Agent prompts and output parser ready!")

Agent prompts and output parser ready!


### 5.3 Run Multi-Agent Evaluation

In [30]:
multi_agent_results = []

for i, original_item in enumerate(data):
    if i in few_shot_index_set:
        continue

    prompt_item = randomized_data[i]
    start_time = time.perf_counter()

    shuffled_agent_votes = []
    raw_outputs = []
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0
    error_messages = []

    for agent_name, focus in agent_roles:
        try:
            prompt = build_agent_prompt(prompt_item, agent_name, focus)
            response = agent_llm.invoke(prompt)
            raw_output = response_to_text(response)
            vote = extract_agent_choice(raw_output)

            input_t, output_t, total_t = get_token_usage(response)
            total_input_tokens += input_t or 0
            total_output_tokens += output_t or 0
            total_tokens += total_t or 0

        except Exception as e:
            vote = "Error"
            raw_output = str(e)
            error_messages.append(str(e))

        shuffled_agent_votes.append(vote)
        raw_outputs.append(raw_output)

    valid_votes = [
        vote for vote in shuffled_agent_votes
        if vote in ["Answer1", "Answer2"]
    ]
    count_1 = valid_votes.count("Answer1")
    count_2 = valid_votes.count("Answer2")

    if count_1 >= 2 and count_1 > count_2:
        shuffled_prediction = "Answer1"
    elif count_2 >= 2 and count_2 > count_1:
        shuffled_prediction = "Answer2"
    else:
        shuffled_prediction = "Unparseable"

    prediction = map_prediction_to_original(
        shuffled_prediction, prompt_item["_was_swapped"]
    )
    agent_votes = [
        map_prediction_to_original(vote, prompt_item["_was_swapped"])
        for vote in shuffled_agent_votes
    ]
    human_preference = original_item["Preference"]
    latency = time.perf_counter() - start_time

    multi_agent_results.append({
        "sample_id": i + 1,
        "human_preference": human_preference,
        "prediction": prediction,
        "shuffled_prediction": shuffled_prediction,
        "answer_order": prompt_item["_answer_order"],
        "correct": prediction == human_preference,
        "agent_1": agent_votes[0],
        "agent_2": agent_votes[1],
        "agent_3": agent_votes[2],
        "votes": " | ".join(agent_votes),
        "shuffled_votes": " | ".join(shuffled_agent_votes),
        "latency_seconds": latency,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "raw_outputs": "\n---\n".join(raw_outputs),
        "error": "; ".join(error_messages) if error_messages else None
    })

    if len(multi_agent_results) % 10 == 0:
        print(
            f"Processed {len(multi_agent_results)}/"
            f"{len(data) - len(few_shot_examples)} samples"
        )

multi_agent_df = pd.DataFrame(multi_agent_results)

print("\nMulti-Agent evaluation completed!")
print("Total test samples:", len(multi_agent_df))


Processed 10/96 samples
Processed 20/96 samples
Processed 30/96 samples
Processed 40/96 samples
Processed 50/96 samples
Processed 60/96 samples
Processed 70/96 samples
Processed 80/96 samples
Processed 90/96 samples

Multi-Agent evaluation completed!
Total test samples: 96


### 5.4 Evaluate Multi-Agent Results

In [31]:
total_samples = len(multi_agent_df)

consistency_rate = multi_agent_df["correct"].mean() * 100

valid_rate = multi_agent_df["prediction"].isin(
    ["Answer1", "Answer2"]
).mean() * 100

average_latency = multi_agent_df["latency_seconds"].mean()

total_input_tokens = multi_agent_df["input_tokens"].sum()
total_output_tokens = multi_agent_df["output_tokens"].sum()
total_tokens = multi_agent_df["total_tokens"].sum()

print("===== Multi-Agent Evaluation Results =====")

print("Total samples:", total_samples)
print(f"Consistency Rate: {consistency_rate:.2f}%")
print(f"Valid Prediction Rate: {valid_rate:.2f}%")
print(f"Average Latency: {average_latency:.3f} seconds")

print("Total Input Tokens:", total_input_tokens)
print("Total Output Tokens:", total_output_tokens)
print("Total Tokens:", total_tokens)

multi_agent_df.to_csv(
    "multi_agent_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to multi_agent_results.csv")

===== Multi-Agent Evaluation Results =====
Total samples: 96
Consistency Rate: 64.58%
Valid Prediction Rate: 100.00%
Average Latency: 5.218 seconds
Total Input Tokens: 80877
Total Output Tokens: 37234
Total Tokens: 118111

Results saved to multi_agent_results.csv


### 5.5 Compare All Prompting Methods

In [32]:
comparison_df = pd.DataFrame([
    {
        "Method": "Baseline",
        "Consistency Rate": baseline_test_df["correct"].mean() * 100,
        "Average Latency": baseline_test_df["latency_seconds"].mean(),
        "Total Tokens": baseline_test_df["total_tokens"].sum()
    },
    {
        "Method": "Few-shot",
        "Consistency Rate": few_shot_df["correct"].mean() * 100,
        "Average Latency": few_shot_df["latency_seconds"].mean(),
        "Total Tokens": few_shot_df["total_tokens"].sum()
    },
    {
        "Method": "Chain-of-Thought",
        "Consistency Rate": cot_df["correct"].mean() * 100,
        "Average Latency": cot_df["latency_seconds"].mean(),
        "Total Tokens": cot_df["total_tokens"].sum()
    },
    {
        "Method": "Self-Consistency",
        "Consistency Rate": sc_df["correct"].mean() * 100,
        "Average Latency": sc_df["latency_seconds"].mean(),
        "Total Tokens": sc_df["total_tokens"].sum()
    },
    {
        "Method": "Multi-Agent",
        "Consistency Rate": multi_agent_df["correct"].mean() * 100,
        "Average Latency": multi_agent_df["latency_seconds"].mean(),
        "Total Tokens": multi_agent_df["total_tokens"].sum()
    }
])

display(comparison_df)

,Method,Consistency Rate,Average Latency,Total Tokens
0,Baseline,63.541667,0.938827,24079
1,Few-shot,66.666667,0.976652,86287
2,Chain-of-Thought,63.541667,2.715534,54208
3,Self-Consistency,64.583333,8.658356,181616
4,Multi-Agent,64.583333,5.218277,118111


## 6. Additional Analysis

### 6.1 Paired Error Analysis

In [33]:
from scipy.stats import binomtest

cot_compare = cot_df[
    ["sample_id", "correct"]
].rename(columns={"correct": "cot_correct"})

multi_compare = multi_agent_df[
    ["sample_id", "correct"]
].rename(columns={"correct": "multi_agent_correct"})

paired_df = cot_compare.merge(
    multi_compare,
    on="sample_id",
    validate="one_to_one"
)

multi_only = (
    (paired_df["multi_agent_correct"]) &
    (~paired_df["cot_correct"])
).sum()

cot_only = (
    (paired_df["cot_correct"]) &
    (~paired_df["multi_agent_correct"])
).sum()

both_correct = (
    paired_df["cot_correct"] &
    paired_df["multi_agent_correct"]
).sum()

both_wrong = (
    (~paired_df["cot_correct"]) &
    (~paired_df["multi_agent_correct"])
).sum()

n_discordant = int(multi_only + cot_only)

p_value = (
    binomtest(
        int(multi_only),
        n=n_discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue
    if n_discordant > 0 else 1.0
)

print("===== Paired Error Analysis =====")

print("Total matched samples:", len(paired_df))
print("Both correct:", both_correct)
print("Both wrong:", both_wrong)
print("Multi-Agent only correct:", multi_only)
print("CoT only correct:", cot_only)

print(f"\nMcNemar exact test p-value: {p_value:.4f}")

print("\nCases improved by Multi-Agent:")
print(
    paired_df.loc[
        paired_df["multi_agent_correct"] &
        (~paired_df["cot_correct"]),
        "sample_id"
    ].tolist()
)

print("\nCases improved by CoT:")
print(
    paired_df.loc[
        paired_df["cot_correct"] &
        (~paired_df["multi_agent_correct"]),
        "sample_id"
    ].tolist()
)

===== Paired Error Analysis =====
Total matched samples: 96
Both correct: 55
Both wrong: 28
Multi-Agent only correct: 7
CoT only correct: 6

McNemar exact test p-value: 1.0000

Cases improved by Multi-Agent:
[4, 6, 17, 35, 36, 65, 95]

Cases improved by CoT:
[7, 25, 38, 81, 82, 84]


### 6.2 Statistical Significance Testing

In [34]:

import numpy as np
import pandas as pd
import itertools

from scipy.stats import (
    chi2,
    binomtest,
    friedmanchisquare,
    wilcoxon
)

# ==========================================
# 6.2.1 Prepare Matched Experimental Data
# ==========================================

baseline_test_df = baseline_df[
    baseline_df["sample_id"].isin(
        few_shot_df["sample_id"]
    )
].copy()

method_dfs = {
    "Baseline": baseline_test_df,
    "Few-shot": few_shot_df,
    "Chain-of-Thought": cot_df,
    "Self-Consistency": sc_df,
    "Multi-Agent": multi_agent_df
}

required_columns = [
    "sample_id",
    "correct",
    "latency_seconds",
    "total_tokens"
]

reference_ids = baseline_test_df["sample_id"].tolist()

if len(reference_ids) != len(set(reference_ids)):
    raise ValueError("Duplicate sample IDs in Baseline.")

for method, df in method_dfs.items():

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{method} missing columns: {missing}"
        )

    if df["sample_id"].duplicated().any():
        raise ValueError(
            f"Duplicate sample IDs in {method}."
        )

    if set(df["sample_id"]) != set(reference_ids):
        raise ValueError(
            f"Sample IDs do not match in {method}."
        )

aligned = {
    method: df.set_index("sample_id").loc[reference_ids]
    for method, df in method_dfs.items()
}

accuracy_matrix = pd.DataFrame({
    method: aligned[method]["correct"].astype(int)
    for method in method_dfs
})

latency_matrix = pd.DataFrame({
    method: pd.to_numeric(
        aligned[method]["latency_seconds"],
        errors="coerce"
    )
    for method in method_dfs
})

tokens_matrix = pd.DataFrame({
    method: pd.to_numeric(
        aligned[method]["total_tokens"],
        errors="coerce"
    )
    for method in method_dfs
})

if not accuracy_matrix.isin([0, 1]).all().all():
    raise ValueError("Correctness must contain only 0 or 1.")

for name, matrix in [
    ("Latency", latency_matrix),
    ("Tokens", tokens_matrix)
]:
    if matrix.isna().any().any():
        raise ValueError(f"Invalid values in {name}.")

n = len(reference_ids)
k = len(method_dfs)

print("Matched experimental data prepared.")
print("Methods:", k)
print("Matched samples:", n)


# ==========================================
# 6.2.2 Overall Statistical Tests
# ==========================================

accuracy_values = accuracy_matrix.to_numpy()

row_sums = accuracy_values.sum(axis=1)
col_sums = accuracy_values.sum(axis=0)

total = accuracy_values.sum()

denominator = (
    k * total - np.square(row_sums).sum()
)

numerator = (
    (k - 1) *
    (k * np.square(col_sums).sum() - total ** 2)
)

if denominator > 0:
    cochran_q = numerator / denominator
    accuracy_p = chi2.sf(cochran_q, k - 1)
else:
    cochran_q = np.nan
    accuracy_p = np.nan

# Effect Size for Accuracy
accuracy_w = (
    cochran_q / (n * (k - 1))
    if np.isfinite(cochran_q) else np.nan
)

# Friedman Test: Latency
latency_result = friedmanchisquare(
    *[
        latency_matrix[method].to_numpy()
        for method in method_dfs
    ]
)

# Friedman Test: Total Tokens
tokens_result = friedmanchisquare(
    *[
        tokens_matrix[method].to_numpy()
        for method in method_dfs
    ]
)

latency_w = (
    latency_result.statistic / (n * (k - 1))
)

tokens_w = (
    tokens_result.statistic / (n * (k - 1))
)

overall_tests = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "Test": "Cochran's Q",
        "Statistic": cochran_q,
        "p-value": accuracy_p,
        "Effect Size": accuracy_w
    },
    {
        "Metric": "Latency",
        "Test": "Friedman Test",
        "Statistic": latency_result.statistic,
        "p-value": latency_result.pvalue,
        "Effect Size": latency_w
    },
    {
        "Metric": "Total Tokens",
        "Test": "Friedman Test",
        "Statistic": tokens_result.statistic,
        "p-value": tokens_result.pvalue,
        "Effect Size": tokens_w
    }
])

print("\n===== Overall Statistical Tests =====")
display(overall_tests.round(6))


# ==========================================
# 6.2.3 Pairwise Statistical Comparisons
# ==========================================

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (m - np.arange(m)) * sorted_p
    )

    adjusted_sorted = np.minimum(
        adjusted_sorted, 1.0
    )

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted

    return adjusted


def paired_bootstrap_ci(
    differences,
    n_boot=10000,
    seed=42
):

    differences = np.asarray(
        differences, dtype=float
    )

    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        len(differences),
        size=(n_boot, len(differences))
    )

    bootstrap_means = differences[indices].mean(axis=1)

    return np.quantile(
        bootstrap_means,
        [0.025, 0.975]
    )


pairwise_rows = []

methods = list(method_dfs.keys())

comparison_matrices = {
    "Accuracy": accuracy_matrix,
    "Latency": latency_matrix,
    "Total Tokens": tokens_matrix
}

for metric, matrix in comparison_matrices.items():

    for method_a, method_b in itertools.combinations(
        methods, 2
    ):

        a = matrix[method_a].to_numpy()
        b = matrix[method_b].to_numpy()

        differences = a - b

        if metric == "Accuracy":

            a_only = int(
                ((a == 1) & (b == 0)).sum()
            )

            b_only = int(
                ((a == 0) & (b == 1)).sum()
            )

            discordant = a_only + b_only

            if discordant == 0:
                p_value = 1.0
            else:
                p_value = binomtest(
                    a_only,
                    n=discordant,
                    p=0.5,
                    alternative="two-sided"
                ).pvalue

            scale = 100

        else:

            if np.allclose(differences, 0):
                p_value = 1.0
            else:
                p_value = wilcoxon(
                    a,
                    b,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue

            scale = 1

        ci_lower, ci_upper = paired_bootstrap_ci(
            differences
        )

        pairwise_rows.append({
            "Metric": metric,
            "Method A": method_a,
            "Method B": method_b,
            "Mean Difference (A-B)": (
                differences.mean() * scale
            ),
            "95% CI Lower": ci_lower * scale,
            "95% CI Upper": ci_upper * scale,
            "p-value": p_value
        })


pairwise_results = pd.DataFrame(pairwise_rows)


# ==========================================
# 6.2.4 Multiple Comparison Correction
# ==========================================

pairwise_results["Holm-adjusted p"] = np.nan

for metric in pairwise_results["Metric"].unique():

    mask = pairwise_results["Metric"] == metric

    pairwise_results.loc[
        mask, "Holm-adjusted p"
    ] = holm_adjust(
        pairwise_results.loc[
            mask, "p-value"
        ].to_numpy()
    )

pairwise_results["Significant"] = (
    pairwise_results["Holm-adjusted p"] < 0.05
)

print("\n===== Pairwise Statistical Comparisons =====")
display(pairwise_results.round(4))


# ==========================================
# 6.2.5 Export Statistical Results
# ==========================================

overall_tests.to_csv(
    "overall_statistical_tests.csv",
    index=False,
    encoding="utf-8-sig"
)

pairwise_results.to_csv(
    "pairwise_statistical_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nStatistical analysis completed.")

print("Saved files:")
print("1. overall_statistical_tests.csv")
print("2. pairwise_statistical_results.csv")


Matched experimental data prepared.
Methods: 5
Matched samples: 96

===== Overall Statistical Tests =====


,Metric,Test,Statistic,p-value,Effect Size
0,Accuracy,Cochran's Q,0.952381,0.916929,0.002480
1,Latency,Friedman Test,365.208333,0.000000,0.951063
2,Total Tokens,Friedman Test,383.208333,0.000000,0.997938



===== Pairwise Statistical Comparisons =====


,Metric,Method A,Method B,Mean Difference (A-B),95% CI Lower,95% CI Upper,p-value,Holm-adjusted p,Significant
0,Accuracy,Baseline,Few-shot,-3.1250,-10.4167,4.1667,0.5811,1.0000,False
1,Accuracy,Baseline,Chain-of-Thought,0.0000,-8.3333,8.3333,1.0000,1.0000,False
2,Accuracy,Baseline,Self-Consistency,-1.0417,-8.3333,6.2500,1.0000,1.0000,False
3,Accuracy,Baseline,Multi-Agent,-1.0417,-6.2500,4.1667,1.0000,1.0000,False
4,Accuracy,Few-shot,Chain-of-Thought,3.1250,-5.2083,11.4583,0.6476,1.0000,False
5,Accuracy,Few-shot,Self-Consistency,2.0833,-5.2083,9.3750,0.7744,1.0000,False
6,Accuracy,Few-shot,Multi-Agent,2.0833,-5.2083,9.3750,0.7905,1.0000,False
7,Accuracy,Chain-of-Thought,Self-Consistency,-1.0417,-9.3750,7.2917,1.0000,1.0000,False
8,Accuracy,Chain-of-Thought,Multi-Agent,-1.0417,-8.3333,6.2500,1.0000,1.0000,False
9,Accuracy,Self-Consistency,Multi-Agent,0.0000,-5.2083,5.2083,1.0000,1.0000,False



Statistical analysis completed.
Saved files:
1. overall_statistical_tests.csv
2. pairwise_statistical_results.csv


# 6.3 Efficiency and Pareto Analysis

In [35]:
efficiency_rows = []

for method in method_dfs:

    correct_count = accuracy_matrix[method].sum()
    total_tokens = tokens_matrix[method].sum()

    efficiency_rows.append({
        "Method": method,
        "Accuracy (%)": accuracy_matrix[method].mean() * 100,
        "Average Latency (s)": latency_matrix[method].mean(),
        "Average Tokens": tokens_matrix[method].mean(),
        "Total Tokens": total_tokens,
        "Correct per 1000 Tokens": (
            correct_count / total_tokens * 1000
            if total_tokens > 0 else np.nan
        )
    })

efficiency_df = pd.DataFrame(efficiency_rows)

# Identify Pareto-optimal methods
pareto_flags = []

for i, current in efficiency_df.iterrows():

    dominated = False

    for j, other in efficiency_df.iterrows():

        if i == j:
            continue

        no_worse = (
            other["Accuracy (%)"] >= current["Accuracy (%)"]
            and other["Average Latency (s)"] <= current["Average Latency (s)"]
            and other["Total Tokens"] <= current["Total Tokens"]
        )

        strictly_better = (
            other["Accuracy (%)"] > current["Accuracy (%)"]
            or other["Average Latency (s)"] < current["Average Latency (s)"]
            or other["Total Tokens"] < current["Total Tokens"]
        )

        if no_worse and strictly_better:
            dominated = True
            break

    pareto_flags.append(not dominated)

efficiency_df["Pareto Optimal"] = pareto_flags

efficiency_df = efficiency_df.sort_values(
    "Accuracy (%)",
    ascending=False
).reset_index(drop=True)

print("===== Efficiency and Pareto Analysis =====")
display(efficiency_df.round(4))

efficiency_df.to_csv(
    "efficiency_pareto_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: efficiency_pareto_analysis.csv")


===== Efficiency and Pareto Analysis =====


,Method,Accuracy (%),Average Latency (s),Average Tokens,Total Tokens,Correct per 1000 Tokens,Pareto Optimal
0,Few-shot,66.6667,0.9767,898.8229,86287,0.7417,True
1,Multi-Agent,64.5833,5.2183,1230.3229,118111,0.5249,False
2,Self-Consistency,64.5833,8.6584,1891.8333,181616,0.3414,False
3,Baseline,63.5417,0.9388,250.8229,24079,2.5333,True
4,Chain-of-Thought,63.5417,2.7155,564.6667,54208,1.1253,False



Saved: efficiency_pareto_analysis.csv


# 6.4 Ablation Study

In [36]:
valid_labels = ["Answer1", "Answer2"]

ablation_df = multi_agent_df.copy()

if ablation_df["sample_id"].duplicated().any():
    raise ValueError("Duplicate sample IDs.")

if set(ablation_df["sample_id"]) != set(few_shot_df["sample_id"]):
    raise ValueError("Sample IDs do not match.")

agent_columns = ["agent_1", "agent_2", "agent_3"]

for col in agent_columns:
    if col not in ablation_df.columns:
        raise ValueError(f"Missing column: {col}")


def two_agent_vote(row, remaining_agents):

    votes = [row[col] for col in remaining_agents]

    if (
        all(vote in valid_labels for vote in votes)
        and votes[0] == votes[1]
    ):
        return votes[0]

    return "Unparseable"


ablation_methods = {
    "Full Multi-Agent": None,
    "Without Accuracy Evaluator": [
        "agent_2", "agent_3"
    ],
    "Without Helpfulness Evaluator": [
        "agent_1", "agent_3"
    ],
    "Without Quality and Safety Evaluator": [
        "agent_1", "agent_2"
    ]
}

ablation_predictions = pd.DataFrame({
    "sample_id": ablation_df["sample_id"],
    "human_preference": ablation_df["human_preference"]
})

summary_rows = []

for method, remaining_agents in ablation_methods.items():

    if remaining_agents is None:
        predictions = ablation_df["prediction"].copy()
    else:
        predictions = ablation_df.apply(
            lambda row: two_agent_vote(
                row, remaining_agents
            ),
            axis=1
        )

    valid_mask = predictions.isin(valid_labels)

    accuracy = (
        predictions == ablation_df["human_preference"]
    ).mean() * 100

    valid_rate = valid_mask.mean() * 100

    valid_accuracy = (
        (
            predictions[valid_mask]
            == ablation_df.loc[valid_mask, "human_preference"]
        ).mean() * 100
        if valid_mask.any() else float("nan")
    )

    summary_rows.append({
        "Method": method,
        "Accuracy (%)": accuracy,
        "Valid Prediction Rate (%)": valid_rate,
        "Accuracy on Valid Predictions (%)": valid_accuracy,
        "Correct Samples": int(
            (
                predictions
                == ablation_df["human_preference"]
            ).sum()
        ),
        "Unresolved Samples": int((~valid_mask).sum())
    })

    ablation_predictions[method] = predictions


ablation_results = pd.DataFrame(summary_rows)

full_accuracy = ablation_results.loc[
    ablation_results["Method"] == "Full Multi-Agent",
    "Accuracy (%)"
].iloc[0]

ablation_results["Accuracy Change (pp)"] = (
    ablation_results["Accuracy (%)"] - full_accuracy
)

print("===== Ablation Study Results =====")

display(ablation_results.round(4))

ablation_results.to_csv(
    "ablation_study_results.csv",
    index=False,
    encoding="utf-8-sig"
)

ablation_predictions.to_csv(
    "ablation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved files:")
print("1. ablation_study_results.csv")
print("2. ablation_predictions.csv")


===== Ablation Study Results =====


,Method,Accuracy (%),Valid Prediction Rate (%),Accuracy on Valid Predictions (%),Correct Samples,Unresolved Samples,Accuracy Change (pp)
0,Full Multi-Agent,64.5833,100.0000,64.5833,62,0,0.0000
1,Without Accuracy Evaluator,60.4167,88.5417,68.2353,58,11,-4.1667
2,Without Helpfulness Evaluator,58.3333,87.5000,66.6667,56,12,-6.2500
3,Without Quality and Safety Evaluator,58.3333,88.5417,65.8824,56,11,-6.2500



Saved files:
1. ablation_study_results.csv
2. ablation_predictions.csv


# 6.5 Answer Order Bias Analysis

In [39]:
from scipy.stats import fisher_exact

# Reconstruct sample IDs from original row positions
order_df = pd.DataFrame({
    "sample_id": range(1, len(randomized_data) + 1),
    "Answer Order": [
        item["_answer_order"]
        for item in randomized_data
    ]
})

order_df["sample_id"] = order_df["sample_id"].astype(int)

if order_df["Answer Order"].isna().any():
    raise ValueError("Missing answer order information.")

valid_orders = {"Original", "Swapped"}

if not set(order_df["Answer Order"]).issubset(valid_orders):
    raise ValueError("Unexpected answer order labels.")

rows = []

for method, df in method_dfs.items():

    temp = df[["sample_id", "correct"]].copy()
    temp["sample_id"] = temp["sample_id"].astype(int)

    temp = temp.merge(
        order_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )

    if temp["Answer Order"].isna().any():
        raise ValueError(f"Missing order mapping: {method}")

    original = temp[
        temp["Answer Order"] == "Original"
    ]

    swapped = temp[
        temp["Answer Order"] == "Swapped"
    ]

    n_original = len(original)
    n_swapped = len(swapped)

    if n_original == 0 or n_swapped == 0:
        raise ValueError("Empty answer order group.")

    correct_original = int(original["correct"].sum())
    correct_swapped = int(swapped["correct"].sum())

    acc_original = correct_original / n_original * 100
    acc_swapped = correct_swapped / n_swapped * 100

    contingency = [
        [
            correct_original,
            n_original - correct_original
        ],
        [
            correct_swapped,
            n_swapped - correct_swapped
        ]
    ]

    _, p_value = fisher_exact(contingency)

    rows.append({
        "Method": method,
        "Original Samples": n_original,
        "Original Accuracy (%)": acc_original,
        "Swapped Samples": n_swapped,
        "Swapped Accuracy (%)": acc_swapped,
        "Difference (pp)": acc_swapped - acc_original,
        "p-value": p_value
    })


order_results = pd.DataFrame(rows)


# Holm correction

p_values = order_results["p-value"].to_numpy()

m = len(p_values)
indices = np.argsort(p_values)

adjusted_sorted = np.maximum.accumulate(
    (m - np.arange(m)) * p_values[indices]
)

adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

adjusted_p = np.empty(m)
adjusted_p[indices] = adjusted_sorted

order_results["Holm-adjusted p"] = adjusted_p

order_results["Significant"] = (
    order_results["Holm-adjusted p"] < 0.05
)


# Display results

matched_ids = set(
    baseline_test_df["sample_id"].astype(int)
)

matched_orders = order_df[
    order_df["sample_id"].isin(matched_ids)
]

print("===== Answer Order Distribution =====")
print(matched_orders["Answer Order"].value_counts())

print("\n===== Answer Order Bias Analysis =====")
display(order_results.round(4))


# Export results

order_results.to_csv(
    "answer_order_bias_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: answer_order_bias_analysis.csv")


===== Answer Order Distribution =====
Answer Order
Original    49
Swapped     47
Name: count, dtype: int64

===== Answer Order Bias Analysis =====


,Method,Original Samples,Original Accuracy (%),Swapped Samples,Swapped Accuracy (%),Difference (pp),p-value,Holm-adjusted p,Significant
0,Baseline,49,59.1837,47,68.0851,8.9014,0.4021,1.0,False
1,Few-shot,49,61.2245,47,72.3404,11.1159,0.2839,1.0,False
2,Chain-of-Thought,49,63.2653,47,63.8298,0.5645,1.0000,1.0,False
3,Self-Consistency,49,63.2653,47,65.9574,2.6921,0.8331,1.0,False
4,Multi-Agent,49,61.2245,47,68.0851,6.8606,0.5270,1.0,False



Saved: answer_order_bias_analysis.csv


# 6.6 Classification Performance Metrics

In [42]:

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

# ==========================================
# 6.6.1 Classification Performance Metrics
# ==========================================

labels = ["Answer1", "Answer2"]

classification_rows = []
per_class_rows = []
confusion_rows = []

reference_ids = baseline_test_df["sample_id"].tolist()

for method, df in method_dfs.items():

    # Align samples
    if df["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample IDs in {method}.")

    if set(df["sample_id"]) != set(reference_ids):
        raise ValueError(f"Sample IDs do not match in {method}.")

    aligned_df = (
        df.set_index("sample_id")
        .loc[reference_ids]
    )

    # Ground truth and predicted labels
    y_true = (
        aligned_df["human_preference"]
        .astype(str)
        .str.strip()
    )

    y_pred = (
        aligned_df["prediction"]
        .astype(str)
        .str.strip()
    )

    # Validate labels
    if not y_true.isin(labels).all():
        raise ValueError(
            f"Unexpected ground-truth labels in {method}."
        )

    if not y_pred.isin(labels).all():
        raise ValueError(
            f"Unexpected prediction labels in {method}."
        )

    # Validate existing correctness records
    correct_values = (
        aligned_df["correct"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

    if correct_values.isna().any():
        raise ValueError(
            f"Invalid correctness values in {method}."
        )

    expected_correct = y_true.eq(y_pred).to_numpy()

    if not np.array_equal(
        correct_values.to_numpy(dtype=bool),
        expected_correct
    ):
        raise ValueError(
            f"Prediction and correctness mismatch in {method}."
        )

    # Overall classification metrics
    accuracy = accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        )
    )

    _, _, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=labels,
            average="weighted",
            zero_division=0
        )
    )

    classification_rows.append({
        "Method": method,
        "Samples": len(y_true),
        "Accuracy (%)": accuracy * 100,
        "Macro Precision (%)": macro_precision * 100,
        "Macro Recall (%)": macro_recall * 100,
        "Macro F1 (%)": macro_f1 * 100,
        "Weighted F1 (%)": weighted_f1 * 100
    })

    # Per-class metrics
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=labels,
            average=None,
            zero_division=0
        )
    )

    for i, label in enumerate(labels):

        per_class_rows.append({
            "Method": method,
            "Class": label,
            "Precision (%)": precision[i] * 100,
            "Recall (%)": recall[i] * 100,
            "F1-score (%)": f1[i] * 100,
            "Support": support[i]
        })

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    for i, actual_label in enumerate(labels):
        for j, predicted_label in enumerate(labels):

            confusion_rows.append({
                "Method": method,
                "Actual": actual_label,
                "Predicted": predicted_label,
                "Count": cm[i, j]
            })


# ==========================================
# 6.6.2 Display Results
# ==========================================

classification_results = pd.DataFrame(
    classification_rows
)

per_class_results = pd.DataFrame(
    per_class_rows
)

confusion_results = pd.DataFrame(
    confusion_rows
)

print("===== Overall Classification Performance =====")

display(
    classification_results.round(4)
)

print("\n===== Per-Class Classification Performance =====")

display(
    per_class_results.round(4)
)

print("\n===== Confusion Matrix Results =====")

display(confusion_results)


# ==========================================
# 6.6.3 Export Results
# ==========================================

classification_results.to_csv(
    "classification_performance_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

per_class_results.to_csv(
    "classification_per_class_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

confusion_results.to_csv(
    "classification_confusion_matrices.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nClassification analysis completed.")

print("Saved files:")
print("1. classification_performance_metrics.csv")
print("2. classification_per_class_metrics.csv")
print("3. classification_confusion_matrices.csv")


===== Overall Classification Performance =====


,Method,Samples,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Weighted F1 (%)
0,Baseline,96,63.5417,63.0952,63.0320,63.0566,63.4976
1,Few-shot,96,66.6667,66.2528,66.0816,66.1376,66.5785
2,Chain-of-Thought,96,63.5417,63.0357,62.8126,62.8606,63.3845
3,Self-Consistency,96,64.5833,64.1319,63.5366,63.5714,64.2039
4,Multi-Agent,96,64.5833,64.1026,63.7560,63.8137,64.3635



===== Per-Class Classification Performance =====


,Method,Class,Precision (%),Recall (%),F1-score (%),Support
0,Baseline,Answer1,66.6667,67.9245,67.2897,53
1,Baseline,Answer2,59.5238,58.1395,58.8235,43
2,Few-shot,Answer1,69.0909,71.6981,70.3704,53
3,Few-shot,Answer2,63.4146,60.4651,61.9048,43
4,Chain-of-Thought,Answer1,66.0714,69.8113,67.8899,53
5,Chain-of-Thought,Answer2,60.0000,55.8140,57.8313,43
6,Self-Consistency,Answer1,66.1017,73.5849,69.6429,53
7,Self-Consistency,Answer2,62.1622,53.4884,57.5000,43
8,Multi-Agent,Answer1,66.6667,71.6981,69.0909,53
9,Multi-Agent,Answer2,61.5385,55.8140,58.5366,43



===== Confusion Matrix Results =====


,Method,Actual,Predicted,Count
0,Baseline,Answer1,Answer1,36
1,Baseline,Answer1,Answer2,17
2,Baseline,Answer2,Answer1,18
3,Baseline,Answer2,Answer2,25
4,Few-shot,Answer1,Answer1,38
5,Few-shot,Answer1,Answer2,15
6,Few-shot,Answer2,Answer1,17
7,Few-shot,Answer2,Answer2,26
8,Chain-of-Thought,Answer1,Answer1,37
9,Chain-of-Thought,Answer1,Answer2,16



Classification analysis completed.
Saved files:
1. classification_performance_metrics.csv
2. classification_per_class_metrics.csv
3. classification_confusion_matrices.csv


# 6.7 Representative Case Studies

In [43]:
method_frames = {
    "Baseline": baseline_test_df,
    "Few-shot": few_shot_df,
    "Chain-of-Thought": cot_df,
    "Self-Consistency": sc_df,
    "Multi-Agent": multi_agent_df
}

# 1. Align all methods using the same sample IDs

case_table = baseline_test_df[
    ["sample_id", "human_preference", "answer_order"]
].copy()

for method, df in method_frames.items():

    temp = df[
        ["sample_id", "prediction"]
    ].rename(columns={
        "prediction": method
    })

    case_table = case_table.merge(
        temp,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )

pred_cols = list(method_frames.keys())

# 2. Calculate agreement with human preferences

case_table["correct_count"] = case_table.apply(
    lambda row: sum(
        row[method] == row["human_preference"]
        for method in pred_cols
    ),
    axis=1
)

case_table["prediction_diversity"] = (
    case_table[pred_cols].nunique(axis=1)
)

# 3. Select Case 1: Consensus and correct

consensus_pool = case_table[
    (case_table["correct_count"] == 5) &
    (case_table["prediction_diversity"] == 1)
]

if not consensus_pool.empty:

    case1 = consensus_pool.sort_values(
        "sample_id"
    ).iloc[0]

    print("Case 1: All five methods agree and are correct.")

else:

    case1 = case_table.sort_values(
        ["correct_count", "prediction_diversity", "sample_id"],
        ascending=[False, True, True]
    ).iloc[0]

    print("Case 1: No complete consensus found.")
    print("Showing the case with the highest agreement instead.")

# 4. Select Case 2: Clear disagreement

divergent_pool = case_table[
    (case_table["prediction_diversity"] > 1) &
    (case_table["correct_count"].between(1, 4))
].copy()

if not divergent_pool.empty:

    divergent_pool["vote_balance"] = (
        divergent_pool[pred_cols].apply(
            lambda row: min(
                (row == "Answer1").sum(),
                (row == "Answer2").sum()
            ),
            axis=1
        )
    )

    divergent_pool["correct_balance"] = (
        divergent_pool["correct_count"] - 2.5
    ).abs()

    case2 = divergent_pool.sort_values(
        ["vote_balance", "correct_balance", "sample_id"],
        ascending=[False, True, True]
    ).iloc[0]

else:

    case2 = case_table.sort_values(
        ["prediction_diversity", "sample_id"],
        ascending=[False, True]
    ).iloc[0]

# 5. Display complete case information

def show_case(row, title):

    sample_id = int(row["sample_id"])
    item = data[sample_id - 1]

    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)

    print("\nSample ID:", sample_id)
    print("Answer order:", row["answer_order"])

    print("\nQuestion:")
    print(item["Question"])

    print("\nOriginal Answer 1:")
    print(item["Answer1"])

    print("\nOriginal Answer 2:")
    print(item["Answer2"])

    print("\nGround Truth:")
    print(item["Preference"])

    print("\nPredictions from Five Methods:")

    for method, df in method_frames.items():

        record = df.loc[
            df["sample_id"] == sample_id
        ].iloc[0]

        prediction = record["prediction"]

        print(
            f"{method}: {prediction} | "
            f"Correct: {prediction == item['Preference']}"
        )

        if method in ["Self-Consistency", "Multi-Agent"]:
            if "votes" in record.index:
                print("  Individual votes:", record["votes"])

    # Display reasoning evidence

    cot_record = cot_df.loc[
        cot_df["sample_id"] == sample_id
    ].iloc[0]

    agent_record = multi_agent_df.loc[
        multi_agent_df["sample_id"] == sample_id
    ].iloc[0]

    print("\nCoT Model Explanation:")
    print(str(cot_record.get("raw_output", ""))[:1500])

    print("\nMulti-Agent Evaluations:")
    print(str(agent_record.get("raw_outputs", ""))[:2000])


# 6. Print both cases

show_case(case1, "CASE 1: CONSENSUS CASE")

show_case(case2, "CASE 2: DISAGREEMENT CASE")


Case 1: All five methods agree and are correct.

CASE 1: CONSENSUS CASE

Sample ID: 2
Answer order: Swapped

Question:
Human: I want to learn how to make a New York style cheesecake.  Can you help me?

Original Answer 1:
Assistant: I can help! Let's get started.  Can you describe a New York style cheesecake for me?

Original Answer 2:
Assistant: Sure, I'd love to help.  I've read about cheesecake, and if I'm understanding correctly, you're looking to make one with fresh cheese that's been baked into a crisp crust?  So first, I was going to suggest going to the bookstore to find a recipe that includes detailed instructions, as well as maybe a video of someone demonstrating how to make it.  What do you think?

Ground Truth:
Answer1

Predictions from Five Methods:
Baseline: Answer1 | Correct: True
Few-shot: Answer1 | Correct: True
Chain-of-Thought: Answer1 | Correct: True
Self-Consistency: Answer1 | Correct: True
  Individual votes: Answer1 | Answer1 | Answer2 | Answer2 | Answer1
Multi-Ag